# 🎤 Faster-Whisper STT Server
## AI Voice Assistant — Remote Transcription API

This notebook:
1. Installs Faster-Whisper + Flask + pyngrok
2. Loads `deepdml/faster-whisper-large-v3-turbo-ct2` on the Colab **GPU**
3. Exposes a `POST /transcribe` endpoint via a public **ngrok** URL

### How to use
1. **Runtime → Change runtime type → T4 GPU** (or A100 if available)
2. Run **Cell 1** (install packages)
3. Run **Cell 2** (paste your ngrok auth token)
4. Run **Cell 3** (start the server — copy the `https://...ngrok-free.app` URL)
5. Paste that URL into your local `.env` file as `STT_API_URL=<url>/transcribe`

---
> **Keep the notebook running.** If Colab disconnects, re-run Cells 2–3 and update your `.env`.

In [ ]:
# -- Cell 1: Install dependencies ------------------------------------------
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'faster-whisper', 'ctranslate2', 'flask', 'pyngrok'],
    check=True
)

In [ ]:
# ── Cell 2: Configure ngrok auth token ────────────────────────────────
# Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3GVVwR1jvZi289NAF5BsE6oquUg_4npcsz2Thw1UEu2WyTFGJ"  # <-- paste your ngrok auth token here

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "Please set your ngrok auth token in NGROK_AUTH_TOKEN above.\n"
        "Get it free at: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok auth token configured.")

In [ ]:
# ── Cell 3: Load model and start Flask + ngrok server ─────────────────
import os
import re
import time
import threading
import tempfile
import logging

from faster_whisper import WhisperModel
from flask import Flask, request, jsonify
from pyngrok import ngrok

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ── Model configuration ────────────────────────────────────────────────
MODEL_ID     = "deepdml/faster-whisper-large-v3-turbo-ct2"
DEVICE       = "cuda"   # GPU — make sure Colab runtime is set to GPU
COMPUTE_TYPE = "float16"
BEAM_SIZE    = 5
VAD_FILTER   = True

# ── Load model once at startup ─────────────────────────────────────────
print(f"Loading Faster-Whisper model '{MODEL_ID}' on {DEVICE} …")
load_start = time.perf_counter()

model = WhisperModel(MODEL_ID, device=DEVICE, compute_type=COMPUTE_TYPE)

print(f"Model loaded in {time.perf_counter() - load_start:.1f}s")

# ── Text cleaning (mirrors local whisper_engine.py) ────────────────────
def _clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text

# ── Flask API ──────────────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    """Simple health-check so the local app can verify connectivity."""
    return jsonify({"status": "ok", "model": MODEL_ID, "device": DEVICE})

@app.route("/transcribe", methods=["POST"])
def transcribe():
    """
    POST /transcribe
    ----------------
    multipart/form-data:
        audio  — any audio file Faster-Whisper supports (wav, mp3, webm, ogg …)

    Response (JSON):
        {
            "text":                 str,
            "raw_text":             str,
            "language":             str,
            "language_probability": float,
            "duration":             float,
            "processing_time":      float,
            "segments":             list
        }
    """
    if "audio" not in request.files:
        return jsonify({"error": "No audio file provided. Use multipart/form-data with key 'audio'."}), 400

    audio_file = request.files["audio"]
    filename   = audio_file.filename or "audio"

    # Determine file extension for the temp file
    suffix = ".wav"
    if "." in filename:
        suffix = "." + filename.rsplit(".", 1)[1].lower()

    # Save to a temp file (Faster-Whisper needs a file path)
    fd, tmp_path = tempfile.mkstemp(suffix=suffix)
    os.close(fd)

    try:
        audio_file.save(tmp_path)
        file_size_kb = os.path.getsize(tmp_path) / 1024
        logger.info("Received audio: %s (%.1f KB)", filename, file_size_kb)

        t_start = time.perf_counter()

        segments_gen, info = model.transcribe(
            tmp_path,
            beam_size=BEAM_SIZE,
            vad_filter=VAD_FILTER,
        )

        segments = []
        raw_parts = []
        for seg in segments_gen:
            segments.append({
                "id":    seg.id,
                "start": round(seg.start, 3),
                "end":   round(seg.end, 3),
                "text":  seg.text,
            })
            raw_parts.append(seg.text)

        processing_time = time.perf_counter() - t_start
        raw_text = " ".join(raw_parts).strip()
        cleaned  = _clean_text(raw_text)

        result = {
            "text":                 cleaned,
            "raw_text":             raw_text,
            "language":             info.language,
            "language_probability": round(info.language_probability, 4),
            "duration":             round(info.duration, 3),
            "processing_time":      round(processing_time, 3),
            "segments":             segments,
        }

        logger.info(
            "Transcribed in %.2fs | lang=%s (%.0f%%) | '%s'",
            processing_time,
            info.language,
            info.language_probability * 100,
            cleaned[:80],
        )
        return jsonify(result)

    except Exception as exc:
        logger.exception("Transcription error")
        return jsonify({"error": f"Transcription failed: {exc}"}), 500

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


# ── Start Flask in a background thread ────────────────────────────────
FLASK_PORT = 5000

def run_flask():
    app.run(host="0.0.0.0", port=FLASK_PORT, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(2)  # wait for Flask to start

# ── Open ngrok tunnel ──────────────────────────────────────────────────
public_url = ngrok.connect(FLASK_PORT, bind_tls=True).public_url

print("\n" + "=" * 60)
print(f"  STT API is LIVE at: {public_url}")
print(f"  Transcribe endpoint: {public_url}/transcribe")
print(f"  Health check:        {public_url}/health")
print("=" * 60)
print()
print("Copy the URL above and set it in your local .env:")
print(f"  STT_API_URL={public_url}/transcribe")
print(f"  STT_USE_REMOTE=true")
print()
print("Keep this notebook running. Restart Cells 2-3 if disconnected.")

## Testing the API

You can test the endpoint directly from Colab with the cell below, or use `curl` locally:

```bash
curl -X POST <your-ngrok-url>/transcribe \
  -F 'audio=@/path/to/your/audio.wav' | python -m json.tool
```

In [ ]:
# ── Cell 4 (optional): Quick self-test ────────────────────────────────
# Downloads a short public-domain WAV and round-trips it through the API.
import requests
import urllib.request

TEST_WAV_URL = "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3"

# Use a local file if you have one:
# test_path = "/content/my_audio.wav"

test_path = "/tmp/test_audio.mp3"
urllib.request.urlretrieve(TEST_WAV_URL, test_path)

with open(test_path, "rb") as f:
    resp = requests.post(f"http://localhost:{FLASK_PORT}/transcribe", files={"audio": f})

print(resp.status_code)
import json; print(json.dumps(resp.json(), indent=2))